# CH4: Feast + MinIO + Redis
터미널에서 `docker compose -f experiments/feature_store/compose.yaml up -d --wait`로 전용 서버를 실행한 후, 아래 셀을 순서대로 실행하세요. 커널은 **Python (chapter-4 venv)**입니다.

이 노트북은 원천 준비·정의 등록·과거 조회·적재·온라인 조회를 한 단계씩 수행합니다. 모델 학습은 하지 않습니다.

In [ ]:
from pathlib import Path
import sys, subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/feature_store/run_lab.py").exists())
print(sys.executable)
def run_step(step):
    subprocess.run([sys.executable, "-u", str(root / "experiments/feature_store/run_lab.py"), step], check=True)


## 1. MinIO에 원천 파일 업로드: 버킷 생성 및 파일 해시 검증

In [ ]:
run_step("prepare")


## 2. 피처 정의 등록: Registry 생성. 최초 실행에서는 Redis 키가 0개

In [ ]:
run_step("apply")


## 3. 과거 시점 조회: 원천과 일치하는지, 미래 값이 섞이지 않는지 확인

In [ ]:
run_step("historical")


## 4. Redis에 피처 값 적재: 원천 데이터의 전체 시간 범위 사용

In [ ]:
run_step("materialize")


## 5. 온라인 조회: Redis에서 사용자별 값을 읽고 원천과 비교

In [ ]:
run_step("online")


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(root / "experiments/outputs/feature-store/historical.csv"))
display(pd.read_csv(root / "experiments/outputs/feature-store/online.csv"))


## 비교할 질문
- `apply`와 `materialize`는 각각 무엇을 저장했나요?
- 과거 조회에는 시각이 필요한데 온라인 조회에는 왜 사용자 ID만 전달하나요?
- MinIO의 데이터를 바꿨지만 materialize를 실행하지 않으면 Redis에는 어떤 값이 남을까요?
- 기존 MLflow 모델은 아직 Feast를 이용해 학습한 모델이 아니라는 점을 설명할 수 있나요?